# **Simple RM3 Optimization - Java Fixed**

Focus on what works: RM3 parameter tuning

Your results showed:
- BM25 baseline: 0.0497
- BM25 + RM3: 0.0516 (+3.85%)
- Neural adds nothing

Let's optimize RM3 to get even better results!

## Cell 1: Install Java 21 (FIXED)

In [ ]:
# Install Java 21 (required for Pyserini)
!apt-get update -qq
!apt-get install -y openjdk-21-jdk-headless -qq > /dev/null

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PATH"] = f"{os.environ['JAVA_HOME']}/bin:{os.environ['PATH']}"

# Verify Java version
!java -version

print("\n✅ Java 21 installed")

## Cell 2: Install Python Packages

In [ ]:
!pip install --upgrade pip -q
!pip install pyserini trectools tabulate pandas numpy tqdm -q

print("✅ Packages installed")

## Cell 3: Load Data

In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np
from tqdm import tqdm
from pyserini.search.lucene import LuceneSearcher
from trectools import TrecQrel, TrecRun, TrecEval
from collections import defaultdict
from tabulate import tabulate

drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/TEXT/FinalProject'
QUERIES_PATH = os.path.join(BASE_PATH, 'queriesROBUST.txt')
QRELS_PATH = os.path.join(BASE_PATH, 'qrels_50_Queries')

queries_df = pd.read_csv(QUERIES_PATH, sep='\t', header=None, names=['qid', 'text'], dtype={'qid': str})
TRAIN_QIDS = sorted(queries_df['qid'].tolist(), key=int)[:50]
QUERIES_DICT = dict(zip(queries_df.qid, queries_df.text))

print(f"✅ Loaded {len(TRAIN_QIDS)} queries")

## Cell 4: Initialize Searcher

In [ ]:
print("Loading ROBUST04 index...")
searcher = LuceneSearcher.from_prebuilt_index('robust04')
qrels = TrecQrel(QRELS_PATH)

print("✅ Searcher initialized")

## Cell 5: Helper Functions

In [ ]:
def get_bm25_scores(query, k=1000, k1=0.6, b=0.4):
    """BM25 retrieval"""
    searcher.set_bm25(k1=k1, b=b)
    hits = searcher.search(query, k=k)
    return {h.docid: float(h.score) for h in hits}

def get_rm3_scores(query, k=1000, fb_docs=10, fb_terms=10, alpha=0.5, k1=0.6, b=0.4):
    """BM25 + RM3"""
    searcher.set_bm25(k1=k1, b=b)
    searcher.set_rm3(fb_docs=fb_docs, fb_terms=fb_terms, original_query_weight=alpha)
    hits = searcher.search(query, k=k)
    searcher.unset_rm3()
    return {h.docid: float(h.score) for h in hits}

def evaluate_pipeline(pipeline_func, pipeline_name, queries_dict, qrels):
    """Evaluate a pipeline"""
    all_rows = []
    
    for qid, query_text in tqdm(queries_dict.items(), desc=pipeline_name):
        try:
            scores = pipeline_func(query_text)
            ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:1000]
            
            for rank, (docid, score) in enumerate(ranked, start=1):
                all_rows.append([qid, "Q0", docid, rank, score, pipeline_name])
        except Exception as e:
            print(f"Error on {qid}: {e}")
            continue
    
    run_file = f"{pipeline_name}_run.txt"
    pd.DataFrame(all_rows).to_csv(run_file, sep=' ', header=False, index=False)
    return TrecEval(TrecRun(run_file), qrels)

print("✅ Functions ready")

## Cell 6: Baseline

In [ ]:
print("\nEvaluating baseline...\n")

baseline_te = evaluate_pipeline(
    lambda q: get_bm25_scores(q, k1=0.6, b=0.4),
    "baseline_bm25",
    QUERIES_DICT,
    qrels
)

baseline_map = baseline_te.get_map()
print(f"\nBaseline MAP: {baseline_map:.4f}")

## Cell 7: RM3 Parameter Grid Search

In [ ]:
print("\n" + "="*80)
print("RM3 PARAMETER GRID SEARCH")
print("="*80)

results = []

# Grid search
configs = [
    # (fb_docs, fb_terms, alpha)
    (5, 5, 0.3), (5, 5, 0.5), (5, 5, 0.7),
    (5, 10, 0.3), (5, 10, 0.5), (5, 10, 0.7),
    (5, 20, 0.3), (5, 20, 0.5), (5, 20, 0.7),
    (10, 5, 0.3), (10, 5, 0.5), (10, 5, 0.7),
    (10, 10, 0.3), (10, 10, 0.5), (10, 10, 0.7), (10, 10, 0.9),  # Include default
    (10, 20, 0.3), (10, 20, 0.5), (10, 20, 0.7),
    (10, 30, 0.3), (10, 30, 0.5), (10, 30, 0.7),
    (20, 10, 0.3), (20, 10, 0.5), (20, 10, 0.7),
    (20, 20, 0.3), (20, 20, 0.5), (20, 20, 0.7),
    (20, 30, 0.3), (20, 30, 0.5), (20, 30, 0.7),
]

for fb_docs, fb_terms, alpha in configs:
    name = f"RM3_fd{fb_docs}_ft{fb_terms}_a{alpha}"
    
    pipeline = lambda q, fd=fb_docs, ft=fb_terms, a=alpha: get_rm3_scores(
        q, fb_docs=fd, fb_terms=ft, alpha=a, k1=0.6, b=0.4
    )
    
    te = evaluate_pipeline(pipeline, name, QUERIES_DICT, qrels)
    map_score = te.get_map()
    p5 = te.get_precision(depth=5)
    p10 = te.get_precision(depth=10)
    
    improvement = ((map_score - baseline_map) / baseline_map) * 100
    
    results.append({
        'config': f"fd={fb_docs}, ft={fb_terms}, α={alpha}",
        'map': map_score,
        'p5': p5,
        'p10': p10,
        'improvement': improvement,
        'params': (fb_docs, fb_terms, alpha)
    })
    
    print(f"{name}: MAP={map_score:.4f} ({improvement:+.2f}%)")

# Sort by MAP
results.sort(key=lambda x: x['map'], reverse=True)

print("\n" + "="*80)
print("TOP 10 RM3 CONFIGURATIONS")
print("="*80)

top_results = [[r['config'], f"{r['map']:.4f}", f"{r['improvement']:+.2f}%"] for r in results[:10]]
print(tabulate(top_results, headers=["Configuration", "MAP", "vs Baseline"], tablefmt="fancy_grid"))

best = results[0]
print(f"\n🏆 BEST RM3 CONFIG:")
print(f"   {best['config']}")
print(f"   MAP: {best['map']:.4f} ({best['improvement']:+.2f}% improvement)")
print(f"   P@5: {best['p5']:.4f}")
print(f"   P@10: {best['p10']:.4f}")
print("="*80)

## Cell 8: BM25 Parameter Tuning with Best RM3

In [ ]:
print("\n" + "="*80)
print("BM25 PARAMETER TUNING (with best RM3)")
print("="*80)

best_rm3_params = best['params']
print(f"Using RM3: fb_docs={best_rm3_params[0]}, fb_terms={best_rm3_params[1]}, alpha={best_rm3_params[2]}\n")

bm25_results = []

# Test different BM25 parameters
bm25_configs = [
    (0.4, 0.3), (0.4, 0.4), (0.4, 0.5),
    (0.6, 0.3), (0.6, 0.4), (0.6, 0.5), (0.6, 0.6),
    (0.9, 0.3), (0.9, 0.4), (0.9, 0.5), (0.9, 0.6),
    (1.2, 0.5), (1.2, 0.6), (1.2, 0.75),
    (1.5, 0.5), (1.5, 0.6),
]

for k1, b in bm25_configs:
    name = f"BM25_k{k1}_b{b}_RM3"
    
    pipeline = lambda q, k1_=k1, b_=b: get_rm3_scores(
        q,
        fb_docs=best_rm3_params[0],
        fb_terms=best_rm3_params[1],
        alpha=best_rm3_params[2],
        k1=k1_,
        b=b_
    )
    
    te = evaluate_pipeline(pipeline, name, QUERIES_DICT, qrels)
    map_score = te.get_map()
    
    improvement = ((map_score - baseline_map) / baseline_map) * 100
    
    bm25_results.append({
        'config': f"k1={k1}, b={b}",
        'map': map_score,
        'improvement': improvement,
        'k1': k1,
        'b': b
    })
    
    print(f"{name}: MAP={map_score:.4f} ({improvement:+.2f}%)")

# Sort by MAP
bm25_results.sort(key=lambda x: x['map'], reverse=True)

print("\n" + "="*80)
print("TOP 10 BM25+RM3 CONFIGURATIONS")
print("="*80)

top_bm25 = [[r['config'], f"{r['map']:.4f}", f"{r['improvement']:+.2f}%"] for r in bm25_results[:10]]
print(tabulate(top_bm25, headers=["BM25 Config", "MAP", "vs Baseline"], tablefmt="fancy_grid"))

best_overall = bm25_results[0]
print(f"\n🏆 BEST OVERALL CONFIG:")
print(f"   BM25: k1={best_overall['k1']}, b={best_overall['b']}")
print(f"   RM3: fb_docs={best_rm3_params[0]}, fb_terms={best_rm3_params[1]}, alpha={best_rm3_params[2]}")
print(f"   MAP: {best_overall['map']:.4f} ({best_overall['improvement']:+.2f}% improvement)")
print("="*80)

## Cell 9: Save Best Run

In [ ]:
# Copy best run file
best_run_name = f"BM25_k{best_overall['k1']}_b{best_overall['b']}_RM3_run.txt"
output_file = os.path.join(BASE_PATH, 'final_optimized_run.txt')

!cp {best_run_name} {output_file}

print("\n" + "="*80)
print("FINAL RESULTS")
print("="*80)
print(f"Strategy: RM3 Parameter Optimization")
print(f"Baseline MAP: {baseline_map:.4f}")
print(f"Optimized MAP: {best_overall['map']:.4f}")
print(f"Improvement: {best_overall['improvement']:+.2f}%")
print(f"\n✅ Saved to: {output_file}")
print("="*80)